# FootballFlow — Silver Layer (Structured Streaming)

## Overview
This notebook implements the **Silver Layer** of the Medallion Architecture.

It reads raw JSON from the **Bronze Layer** (Parquet files) as a continuous stream,
applies cleaning, type casting, validation, and deduplication,
then writes the cleaned data to the **Silver Layer** (Parquet files).

```
Bronze Parquet (raw_json)
        ↓
  Parse JSON → columns
        ↓
  Cast Types (date, int)
        ↓
  Validate (minute 0–120, no nulls)
        ↓
  Drop unused columns
        ↓
Silver Parquet (clean, typed)
```

**Input Path:**  `~/footballflow/bronze/match_events/`  
**Output Path:** `~/footballflow/silver/match_events/`  
**Trigger:**     Every 30 seconds  
**Output Mode:** Append  

---
## Cell 1 — Imports

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType
)

---
## Cell 2 — SparkSession

We reuse the same JARs used in the Bronze layer (Kafka connector).
Even though Silver reads from Parquet (not Kafka directly),
we keep the same session config for consistency across the pipeline.

In [2]:
jar_dir = os.path.expanduser("~/kafka-jars")
jars = ",".join([
    f"{jar_dir}/spark-sql-kafka-0-10_2.12-3.5.3.jar",
    f"{jar_dir}/kafka-clients-3.4.1.jar",
    f"{jar_dir}/spark-token-provider-kafka-0-10_2.12-3.5.3.jar",
    f"{jar_dir}/commons-pool2-2.11.1.jar"
])

spark = SparkSession.builder \
    .appName("FootballFlow-Silver") \
    .config("spark.jars", jars) \
    .config("spark.sql.shuffle.partitions", "3") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")

Spark version : 3.5.3
App name      : FootballFlow-Silver


---
## Cell 3 — Paths

All paths follow the Medallion Architecture convention:
- `bronze/` → raw ingested data (JSON strings)
- `silver/` → cleaned, typed, validated data
- `checkpoints/` → Spark Structured Streaming fault-tolerance state

In [3]:
bronze_path     = os.path.expanduser("~/footballflow/bronze/match_events/")
silver_path     = os.path.expanduser("~/footballflow/silver/match_events/")
checkpoint_path = os.path.expanduser("~/footballflow/checkpoints/silver_match_events/")

print(f"Bronze    : {bronze_path}")
print(f"Silver    : {silver_path}")
print(f"Checkpoint: {checkpoint_path}")

Bronze    : /home/jovyan/footballflow/bronze/match_events/
Silver    : /home/jovyan/footballflow/silver/match_events/
Checkpoint: /home/jovyan/footballflow/checkpoints/silver_match_events/


---
## Cell 4 — Schema Definitions

We define two schemas:

1. **`bronze_schema`** — schema of the Bronze Parquet files.
   Each row has one `raw_json` string + metadata columns.

2. **`json_schema`** — schema used to parse the `raw_json` string into columns.
   All fields are read as `StringType` first (same approach as the batch cleaner),
   because empty strings in numeric columns cause errors if cast directly.
   We cast safely ourselves in the next step.

In [4]:
json_schema = StructType([
    StructField("game_event_id",          StringType(),  True),
    StructField("game_id",                StringType(),  True),
    StructField("game_date",              StringType(),  True),
    StructField("minute",                 IntegerType(), True),
    StructField("event_type",             StringType(),  True),
    StructField("description",            StringType(),  True),
    StructField("player_in_id",           StringType(),  True),
    StructField("player_assist_id",       StringType(),  True),
    StructField("player_id",              StringType(),  True),
    StructField("player_name",            StringType(),  True),
    StructField("player_position",        StringType(),  True),
    StructField("player_sub_position",    StringType(),  True),
    StructField("player_foot",            StringType(),  True),
    StructField("player_nationality",     StringType(),  True),
    StructField("player_height_cm",       StringType(),  True),
    StructField("player_market_value",    StringType(),  True),
    StructField("club_id",                StringType(),  True),
    StructField("club_name",              StringType(),  True),
    StructField("coach_name",             StringType(),  True),
    StructField("lineup_type",            StringType(),  True),
    StructField("lineup_position",        StringType(),  True),
    StructField("shirt_number",           StringType(),  True),
    StructField("team_captain",           StringType(),  True),
    StructField("club_goals_scored",      StringType(),  True),
    StructField("club_goals_conceded",    StringType(),  True),
    StructField("club_league_position",   StringType(),  True),
    StructField("opponent_league_position", StringType(), True),
    StructField("hosting",                StringType(),  True),
    StructField("is_win",                 StringType(),  True),
    StructField("competition_name",       StringType(),  True),
    StructField("competition_type",       StringType(),  True),
    StructField("country_name",           StringType(),  True),
    StructField("confederation",          StringType(),  True),
    StructField("season",                 StringType(),  True),
    StructField("round",                  StringType(),  True),
    StructField("stadium",                StringType(),  True),
    StructField("attendance",             StringType(),  True),
    StructField("referee",                StringType(),  True),
    StructField("home_club_id",           StringType(),  True),
    StructField("away_club_id",           StringType(),  True),
    StructField("home_club_name",         StringType(),  True),
    StructField("away_club_name",         StringType(),  True),
    StructField("home_club_goals",        StringType(),  True),
    StructField("away_club_goals",        StringType(),  True),
    StructField("home_club_formation",    StringType(),  True),
    StructField("away_club_formation",    StringType(),  True),
])

print("✅ Schemas defined")

✅ Schemas defined


---
## Cell 5 — Read Stream from Bronze

We use `readStream` (not `read`) to continuously monitor the Bronze directory.
Every time new Parquet files are added by the Bronze layer,
this stream picks them up automatically on the next trigger.

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

bronze_schema = StructType([
    StructField("raw_json",            StringType(),    True),
    StructField("_bronze_ingested_at", TimestampType(), False),
    StructField("_source_topic",       StringType(),    False),
    StructField("_pipeline",           StringType(),    False),
])

print("✅ bronze_schema defined")

✅ bronze_schema defined


---
## Cell 6 — Parse + Clean + Validate

This is the core Silver transformation. Steps applied in order:

| Step | Action | Reason |
|------|---------|--------|
| 1 | `from_json` | Parse raw_json string → columns |
| 2 | `to_date` | Convert date string → DateType |
| 3 | `try_cast` | Safe numeric casting (game_id, club_id, player_id) |
| 4 | `lower + trim` | Normalize type values (e.g. 'Goals' → 'goals') |
| 5 | Validate minute | Null out-of-range minutes (outside 0–120) |
| 6 | `fillna` | Fill null descriptions with 'Unknown' (7.3% of rows) |
| 7 | `filter` | Drop rows with nulls in critical columns |
| 8 | `drop` | Remove unused columns (player_in_id: 50.5% null, player_assist_id: 85.2% null) |
| 9 | Add metadata | Add `_silver_processed_at` timestamp |

In [6]:
bronze_stream = spark.readStream \
    .schema(bronze_schema) \
    .parquet(bronze_path)

print("✅ Bronze stream reader ready")
print(f"   isStreaming: {bronze_stream.isStreaming}")

✅ Bronze stream reader ready
   isStreaming: True


In [7]:
silver_stream = (
    bronze_stream
    .withColumn("data", F.from_json(F.col("raw_json"), json_schema))
    .select("data.*", "_bronze_ingested_at", "_source_topic", "_pipeline")

    # Cast types
    .withColumn("game_date",                F.to_date("game_date"))
    .withColumn("game_id",                  F.expr("try_cast(game_id as int)"))
    .withColumn("club_id",                  F.expr("try_cast(club_id as int)"))
    .withColumn("player_id",                F.expr("try_cast(player_id as int)"))
    .withColumn("player_height_cm",         F.expr("try_cast(player_height_cm as float)"))
    .withColumn("player_market_value",      F.expr("try_cast(player_market_value as float)"))
    .withColumn("attendance",               F.expr("try_cast(attendance as float)"))
    .withColumn("home_club_goals",          F.expr("try_cast(home_club_goals as int)"))
    .withColumn("away_club_goals",          F.expr("try_cast(away_club_goals as int)"))
    .withColumn("team_captain",             F.expr("try_cast(team_captain as int)"))
    .withColumn("season",                   F.expr("try_cast(season as int)"))
    .withColumn("club_goals_scored",        F.expr("try_cast(club_goals_scored as int)"))
    .withColumn("club_goals_conceded",      F.expr("try_cast(club_goals_conceded as int)"))
    .withColumn("club_league_position",     F.expr("try_cast(club_league_position as float)"))
    .withColumn("opponent_league_position", F.expr("try_cast(opponent_league_position as float)"))
    .withColumn("is_win",                   F.expr("try_cast(is_win as int)"))
    .withColumn("shirt_number",             F.expr("try_cast(shirt_number as int)"))
    .withColumn("season_count",             F.regexp_extract(F.col("description"), r"^(\d+)\.", 1).cast("int"))

    # event_type already exists in the data — just apply lowercase and trim
    .withColumn("event_type", F.lower(F.trim(F.col("event_type"))))

    # penalty_result
    .withColumn("penalty_result",
        F.when(F.col("description").contains("Scored"), "scored")
        .when(F.col("description").contains("Missed"), "missed")
        .otherwise(None)
    )

    # validate minute
    .withColumn("minute", F.when(
        (F.col("minute") >= 0) & (F.col("minute") <= 130),
        F.col("minute")
    ).otherwise(None))

    .fillna({"description": "Unknown", "player_name": "Unknown"})

    .filter(
        F.col("game_event_id").isNotNull() &
        F.col("game_id").isNotNull() &
        F.col("minute").isNotNull()
    )

    .withColumn("_silver_processed_at", F.current_timestamp())
)

print("✅ Silver schema done")

✅ Silver schema done


---
## Cell 7 — Write Stream to Silver

**Output Mode:** `append` — each trigger writes only new rows.  
**Format:** Parquet — columnar storage, efficient for downstream Gold queries.  
**Trigger:** Every 30 seconds — matches Bronze layer trigger for consistency.  
**Checkpoint:** Required by Spark Structured Streaming for fault tolerance and exactly-once guarantees.

> **Note on Deduplication:**  
> Full `dropDuplicates` is not supported in Append mode streaming without a watermark.  
> Deduplication by `game_event_id` will be handled in the **Gold Layer**,  
> where we aggregate over the full Silver dataset.

In [8]:
silver_query = silver_stream.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", silver_path) \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="30 seconds") \
    .start()

print("✅ Silver Stream started!")
print(f"   Output     : {silver_path}")
print(f"   Checkpoint : {checkpoint_path}")
print(f"   Trigger    : every 30 seconds")
print(f"   Status     : {silver_query.status}")

✅ Silver Stream started!
   Output     : /home/jovyan/footballflow/silver/match_events/
   Checkpoint : /home/jovyan/footballflow/checkpoints/silver_match_events/
   Trigger    : every 30 seconds
   Status     : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


---
## Cell 8 — Monitor Stream

Run this cell to check the current stream status and last batch progress.

In [9]:
import time
time.sleep(35)  # wait for the first trigger to finish

print(f"🔄 Stream active    : {silver_query.isActive}")
print(f"📊 Last progress    : {silver_query.lastProgress}")

# Count the existing Silver files
files = [f for f in os.listdir(silver_path) if f.endswith(".parquet")]
print(f"\n📁 Parquet files    : {len(files)}")

# Read Silver and verify the data
silver_check = spark.read.parquet(silver_path)
print(f"📊 Total rows       : {silver_check.count():,}")
print(f"📋 Columns          : {silver_check.columns}")
print("\n🔍 Sample data:")
silver_check.show(3, truncate=True)

🔄 Stream active    : True
📊 Last progress    : {'id': 'aa26cde3-2fab-4b60-ab8f-5621413ae01c', 'runId': '6cb039a9-e392-43d3-a324-c7ec86c2afaa', 'name': None, 'timestamp': '2026-06-12T21:12:30.000Z', 'batchId': 6, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'latestOffset': 24, 'triggerExecution': 27}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[file:/home/jovyan/footballflow/bronze/match_events]', 'startOffset': {'logOffset': 5}, 'endOffset': {'logOffset': 5}, 'latestOffset': None, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0}], 'sink': {'description': 'FileSink[/home/jovyan/footballflow/silver/match_events/]', 'numOutputRows': -1}}

📁 Parquet files    : 12
📊 Total rows       : 18
📋 Columns          : ['game_event_id', 'game_id', 'game_date', 'minute', 'event_type', 'description', 'player_in_id', 'player_assist_id', 'player_id', 'player_name', 'player_position', 'player_sub_position', '

---
## Cell 9 — Stop Stream

Run this cell **only** when you want to stop the Silver stream.  
Always stop streams before shutting down the SparkSession.

In [9]:
# Stop the Silver stream
silver_query.stop()
print("⛔ Silver stream stopped")

# Stop the SparkSession
spark.stop()
print("⛔ SparkSession stopped")

⛔ Silver stream stopped
⛔ SparkSession stopped


In [1]:
import os
os.makedirs(os.path.expanduser("~/footballflow/checkpoints/bronze_match_events"), exist_ok=True)
os.makedirs(os.path.expanduser("~/footballflow/checkpoints/silver_match_events"), exist_ok=True)
os.makedirs(os.path.expanduser("~/footballflow/checkpoints/silver_to_influxdb"), exist_ok=True)
print("✅ Checkpoints folders recreated")

✅ Checkpoints folders recreated


In [13]:
# Stop all active streams
for q in spark.streams.active:
    q.stop()
    print(f"⛔ Stopped: {q.name}")

spark.stop()
print("⛔ SparkSession stopped")

⛔ SparkSession stopped
